# 01 · Data collection

Lap-by-lap race data for the **ground-effect era, 2022–2025**: lap times, tyre compound and
age, stints, pit stops, track status, running position and weather.

**Two sources, one schema**

| seasons | source | why |
|---|---|---|
| 2022 | [FastF1](https://docs.fastf1.dev), reading the official F1 live-timing archive | the reference Python library for F1 timing data |
| 2023–2025 | [OpenF1](https://openf1.org) | while building this project, the live-timing archive began answering anonymous requests with **HTTP 401** (an F1 account is now required). OpenF1 serves the same timing data since 2023, free and unauthenticated |

`f1tyres.openf1` converts OpenF1's laps, stints, pit stops, race-control messages and weather
into exactly the columns FastF1 produces. Everything downstream treats all four seasons the
same way. Both loaders cache every request and stay inside the providers' rate limits.

```bash
python -m f1tyres.data --seasons 2022            # FastF1 (paced, resumable)
python -m f1tyres.openf1 --seasons 2023 2024 2025
python -m f1tyres.data --combine-only            # merge into data/processed/
```

In [1]:
import pandas as pd

from f1tyres.data import combine

pd.set_option("display.max_columns", 40)
tables = combine()
laps, results, events = tables["laps"], tables["results"], tables["events"]
{name: df.shape for name, df in tables.items()}

{'laps': (100950, 30), 'results': (1837, 11), 'events': (92, 6)}

In [2]:
events.groupby(["Season", "Source"]).agg(races=("Round", "count"),
                                         laps=("TotalLaps", "sum")).reset_index()

,Season,Source,races,laps
0,2022,FastF1,22,1295
1,2023,OpenF1,22,1339
2,2024,OpenF1,24,1446
3,2025,OpenF1,24,1445


In [3]:
laps.head()

,Season,Round,EventName,Driver,DriverNumber,Team,LapNumber,LapTime,Stint,Compound,TyreLife,FreshTyre,TrackStatus,IsAccurate,Position,Deleted,Sector1Time,Sector2Time,Sector3Time,SpeedST,LapStartTime,PitIn,PitOut,AirTemp,TrackTemp,Humidity,Rainfall,WindSpeed,TotalLaps,Source
0,2022,1,Bahrain Grand Prix,VER,1,Red Bull Racing,1.0,100.236,1.0,SOFT,4.0,False,1,False,2.0,<NA>,NaN,42.325,24.389,250.0,3754.872,False,False,23.9,29.1,25.0,False,0.5,57,FastF1
1,2022,1,Bahrain Grand Prix,VER,1,Red Bull Racing,2.0,97.880,1.0,SOFT,5.0,False,1,True,2.0,<NA>,31.285,42.269,24.326,296.0,3855.340,False,False,23.8,29.0,26.0,False,0.5,57,FastF1
2,2022,1,Bahrain Grand Prix,VER,1,Red Bull Racing,3.0,98.357,1.0,SOFT,6.0,False,1,True,2.0,<NA>,31.499,42.474,24.384,294.0,3953.220,False,False,23.8,29.0,27.0,False,0.5,57,FastF1
3,2022,1,Bahrain Grand Prix,VER,1,Red Bull Racing,4.0,98.566,1.0,SOFT,7.0,False,1,True,2.0,<NA>,31.342,42.674,24.550,297.0,4051.577,False,False,23.8,28.7,29.0,False,0.4,57,FastF1
4,2022,1,Bahrain Grand Prix,VER,1,Red Bull Racing,5.0,98.877,1.0,SOFT,8.0,False,1,True,2.0,<NA>,31.498,42.854,24.525,293.0,4150.143,False,False,23.8,28.5,33.0,False,0.4,57,FastF1


## Do the two sources look alike?

Same columns, similar coverage. The main difference is that OpenF1 has no lap-accuracy check,
so `IsAccurate` is derived from lap time, pit and lap-1 flags. The cleaning step applies its
own filters anyway (green flag, no pit laps, within 107 % of the stint median).

In [4]:
cols = ["LapTime", "Compound", "TyreLife", "Stint", "Position", "TrackTemp", "SpeedST"]
laps.groupby("Source")[cols].apply(lambda d: d.isna().mean()).round(3)

,LapTime,Compound,TyreLife,Stint,Position,TrackTemp,SpeedST
Source,,,,,,,
FastF1,0.004,0.000,0.022,0.000,0.002,0.0,0.002
OpenF1,0.003,0.002,0.002,0.002,0.003,0.0,0.085


In [5]:
laps.groupby(["Source", "Compound"]).size().unstack(fill_value=0)

Compound,HARD,INTERMEDIATE,MEDIUM,None,SOFT,UNKNOWN,WET
Source,,,,,,,
FastF1,8035,1446,9204,334,4182,0,376
OpenF1,36496,4211,27875,0,8532,2,84
